# 🔍 Missing Child Finder — Free Colab GPU Engine
Given your Intel i5 setup, this notebook offloads the heavy AI Inference entirely to Google Colab's Free T4 GPU.

### Important Prep:
1. Go to **Runtime > Change runtime type** in the top menu.
2. Select **T4 GPU** as the Hardware Accelerator.
3. Run **all** cells below in order.

### Endpoints Exposed:
| Endpoint | Purpose |
|---|---|
| `GET /` | Health check |
| `POST /extract-reference` | Extract 512-dim ArcFace embedding from a child's reference photo |
| `POST /analyze` | Detect & match faces in a CCTV frame against the reference embedding |

In [ ]:
!pip install fastapi uvicorn pydantic numpy opencv-python-headless insightface onnxruntime-gpu pyngrok

In [ ]:
%%writefile server.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import base64, numpy as np, cv2, insightface
from insightface.app import FaceAnalysis
import uvicorn
from pyngrok import ngrok

app = FastAPI(title="MCF GPU Engine")

# ── Model Initialization ──────────────────────────────────────────────
print("Loading RetinaFace & ArcFace Models into GPU...")
face_app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
face_app.prepare(ctx_id=0, det_size=(640, 640))

# ── Request Schemas ───────────────────────────────────────────────────
class ReferenceRequest(BaseModel):
    reference_image_b64: str

class AnalyzeRequest(BaseModel):
    frame_b64: str
    reference_embedding: list[float]  # 512-dim ArcFace vector

# ── Endpoints ──────────────────────────────────────────────────────────
@app.get("/")
def health_check():
    return {"status": "ok", "gpu": True, "model": "buffalo_l"}

@app.post("/extract-reference")
def extract_reference(req: ReferenceRequest):
    """
    Receives a Base64-encoded reference photo of the missing child.
    Returns the 512-dimensional ArcFace embedding for future matching.
    """
    try:
        img_data = base64.b64decode(req.reference_image_b64)
        nparr = np.frombuffer(img_data, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        if img is None:
            raise HTTPException(status_code=400, detail="Could not decode the reference image.")
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Invalid image data: {str(e)}")

    faces = face_app.get(img)
    if not faces:
        raise HTTPException(status_code=400, detail="No face detected (RetinaFace) in the reference photo. Please upload a clear frontal photo.")

    # Return the embedding of the most prominent (largest) face
    largest_face = max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))
    embedding = largest_face.embedding.tolist()
    bbox = largest_face.bbox.tolist()

    return {
        "embedding": embedding,
        "face_detected": True,
        "bbox": bbox,
        "embedding_dim": len(embedding)
    }

@app.post("/analyze")
def analyze_frame(req: AnalyzeRequest):
    """
    Receives a Base64-encoded CCTV frame + reference embedding.
    Returns all matching faces with confidence scores and bounding boxes.
    """
    try:
        img_data = base64.b64decode(req.frame_b64)
        nparr = np.frombuffer(img_data, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        if img is None:
            return {"matches": [], "faces_detected": 0, "error": "Could not decode frame"}
    except Exception:
        return {"matches": [], "faces_detected": 0, "error": "Invalid frame data"}

    faces = face_app.get(img)
    ref_emb = np.array(req.reference_embedding, dtype=np.float32)

    matches = []
    for face in faces:
        sim = np.dot(face.embedding, ref_emb) / (
            np.linalg.norm(face.embedding) * np.linalg.norm(ref_emb)
        )
        if sim > 0.68:
            matches.append({
                "confidence": round(float(sim), 4),
                "bbox": [round(c, 1) for c in face.bbox.tolist()]
            })

    # Sort matches by confidence descending
    matches.sort(key=lambda m: m["confidence"], reverse=True)

    return {
        "matches": matches,
        "faces_detected": len(faces),
        "faces_matched": len(matches)
    }

# ── Ngrok Tunnel Setup ─────────────────────────────────────────────────
ngrok_token = "3C6v4S7HhMVKSUUMc4135dPsIUD_3U8iXKHMAS6mudmortCaZ"
ngrok.set_auth_token(ngrok_token)
public_url = ngrok.connect(8050).public_url

print("\n" + "="*50)
print(f"⚡ YOUR ML_SERVICE_URL IS: {public_url} ⚡")
print("COPY THIS URL AND PASTE IT IN YOUR LOCAL TERMINAL!")
print(f'Example: $env:ML_SERVICE_URL="{public_url}"')
print("="*50 + "\n")

if __name__ == "__main__":
    uvicorn.run("server:app", host="0.0.0.0", port=8050)

In [ ]:
!python server.py